# 02 · Model experiments — trying to beat the baseline
**Owner:** P2 (Data & baseline) · **Goal:** a validation macro-F1 above baseline-01 (0.6640), logged in `experiments.md`.

The baseline in `01 baseline model.ipynb` is word + character TF-IDF with a calibrated LinearSVC. Train-set CV looked almost perfect and validation did not, because train tickets repeat. This notebook **does not use cross-validation**. Every candidate is fit on `train.csv` and scored on `validation.csv`.

What is tried
1. Same features, other linear settings (LinearSVC `C`, one logistic regression)
2. Nearby n-gram ranges
3. Frozen multilingual embeddings (`paraphrase-multilingual-MiniLM-L12-v2`) plus logistic regression

A run is appended to `training/experiments.md`. The file in `model/` is replaced only from the last cell, and only when a **TF-IDF** candidate beats the baseline. An embedding win is real as a score, and it stays unshipped until `app/predictor.py` can load that encoder with no internet.


In [ ]:
import sys, json, hashlib, time, warnings, re
from pathlib import Path
import numpy as np
import pandas as pd
import joblib, sklearn

from sklearn.pipeline import Pipeline, make_union
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score

ROOT = Path.cwd() if (Path.cwd() / "app").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from app.textproc import clean_text, build_input

DATA_DIR  = ROOT / "data"
MODEL_DIR = ROOT / "model"
EXP_PATH  = ROOT / "training" / "experiments.md"

SEED = 42
warnings.filterwarnings("ignore")
print("scikit-learn", sklearn.__version__, "| root:", ROOT)


## 1. Load data and the score to beat
Same loading as the baseline (`keep_default_na=False` so an empty subject stays `""`). The bar is the `baseline-01` row already in `experiments.md`.


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", keep_default_na=False)
val   = pd.read_csv(DATA_DIR / "validation.csv", keep_default_na=False)

for df in (train, val):
    df["model_text"] = [build_input(s, t) for s, t in zip(df.subject, df.text)]

X_tr, y_tr = train.model_text, train.category
X_va, y_va = val.model_text, val.category
print(train.shape, val.shape, "| classes:", y_tr.nunique())

def baseline_bar(path=EXP_PATH):
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.startswith("| baseline-01 "):
            parts = [p.strip() for p in line.strip().strip("|").split("|")]
            return float(parts[4]), float(parts[5])
    raise RuntimeError("baseline-01 row missing from experiments.md")

BASELINE_F1, BASELINE_ACC = baseline_bar()
print(f"score to beat: val macro-F1 {BASELINE_F1:.4f} (acc {BASELINE_ACC:.4f})")


## 2. Fit on train, score on validation, log one row

`log_row` will not write the same id twice. Delete that line in `experiments.md` first if you change a setup and want a new number.


In [ ]:
def make_features(word=(1, 2), char=(2, 5), max_char=300_000):
    return make_union(
        TfidfVectorizer(preprocessor=clean_text, analyzer="word", ngram_range=word,
                        min_df=2, sublinear_tf=True, token_pattern=r"(?u)\b\w+\b"),
        TfidfVectorizer(preprocessor=clean_text, analyzer="char_wb", ngram_range=char,
                        min_df=2, sublinear_tf=True, max_features=max_char),
    )

def make_clf(kind, C):
    if kind == "svc":
        return LinearSVC(C=C, class_weight="balanced")
    if kind == "logreg":
        return LogisticRegression(C=C, max_iter=2000, class_weight="balanced")
    raise ValueError(kind)

def log_row(exp_id, model_version, setup, val_f1, val_acc, notes, cv="n/a"):
    line = (f"| {exp_id} | {model_version} | {setup} | {cv} | "
            f"{val_f1:.4f} | {val_acc:.4f} | {notes} |")
    text = EXP_PATH.read_text(encoding="utf-8") if EXP_PATH.exists() else ""
    if f"| {exp_id} |" in text:
        print(exp_id, "already in experiments.md — left as-is")
        return
    with EXP_PATH.open("a", encoding="utf-8") as fh:
        if not text.endswith("\n"):
            fh.write("\n")
        fh.write(line + "\n")
    print(line)

rows = []

def run_tfidf(exp_id, setup, word, char, kind, C):
    pipe = Pipeline([
        ("features", make_features(word, char)),
        ("clf", make_clf(kind, C)),
    ])
    t0 = time.time()
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_va)
    val_f1 = f1_score(y_va, pred, average="macro")
    val_acc = accuracy_score(y_va, pred)
    secs = round(time.time() - t0, 1)
    beat = "beats baseline" if val_f1 > BASELINE_F1 else "below baseline"
    log_row(exp_id, "not-shipped", setup, val_f1, val_acc, f"{beat}; {secs}s; no CV")
    rows.append({"id": exp_id, "setup": setup, "val_macro_f1": val_f1,
                 "val_accuracy": val_acc, "secs": secs, "pipe": pipe, "shippable": True})
    return pipe


## 3. Other linear models, same features as the baseline

Baseline winner was LinearSVC `C=0.5` with word `(1, 2)` and `char_wb` `(2, 5)`. These four stay on that feature setup and only change the classifier. `C=0.5` is not repeated.


In [ ]:
for exp_id, kind, C in [
    ("exp-02a", "svc", 0.1),
    ("exp-02b", "svc", 1.0),
    ("exp-02c", "svc", 2.0),
    ("exp-02d", "logreg", 4.0),
]:
    label = "LinearSVC" if kind == "svc" else "LogReg"
    run_tfidf(
        exp_id,
        f"word(1,2)+char_wb(2,5) TF-IDF + {label} (C={C}, balanced)",
        word=(1, 2), char=(2, 5), kind=kind, C=C,
    )


## 4. Nearby n-gram ranges

Wider word n-grams, then a tighter character range. Classifier stays LinearSVC `C=0.5`, which is the baseline setting, so a change here is the features.


In [ ]:
run_tfidf(
    "exp-03a",
    "word(1,3)+char_wb(2,5) TF-IDF + LinearSVC (C=0.5, balanced)",
    word=(1, 3), char=(2, 5), kind="svc", C=0.5,
)
run_tfidf(
    "exp-03b",
    "word(1,2)+char_wb(3,5) TF-IDF + LinearSVC (C=0.5, balanced)",
    word=(1, 2), char=(3, 5), kind="svc", C=0.5,
)


## 5. Multilingual embeddings

Char n-grams already cover Sinhala and Tamil without a dictionary. This check asks whether a frozen multilingual encoder does better. It is **not** a fine-tune.

Install once, in the same environment as this notebook:

```
pip install sentence-transformers
```

The first run downloads `paraphrase-multilingual-MiniLM-L12-v2`. Later runs can be offline if that cache is present. Do not put a download in the API container. If this score wins, keep shipping the TF-IDF model until `predictor.py` loads the encoder from disk.


In [ ]:
EMB_ID = "exp-04-minilm"
EMB_SETUP = "paraphrase-multilingual-MiniLM-L12-v2 + LogReg (C=4, balanced), frozen encoder"

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    print("sentence-transformers is not installed. Skipping", EMB_ID)
    print("pip install sentence-transformers   # then re-run this cell")
else:
    t0 = time.time()
    encoder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    tr_text = X_tr.map(clean_text).tolist()
    va_text = X_va.map(clean_text).tolist()
    emb_tr = encoder.encode(tr_text, batch_size=64, show_progress_bar=True)
    emb_va = encoder.encode(va_text, batch_size=64, show_progress_bar=True)
    head = LogisticRegression(C=4, max_iter=2000, class_weight="balanced")
    head.fit(emb_tr, y_tr)
    pred = head.predict(emb_va)
    val_f1 = f1_score(y_va, pred, average="macro")
    val_acc = accuracy_score(y_va, pred)
    secs = round(time.time() - t0, 1)
    beat = "beats baseline" if val_f1 > BASELINE_F1 else "below baseline"
    log_row(EMB_ID, "not-shipped", EMB_SETUP, val_f1, val_acc,
            f"{beat}; {secs}s; encoder not in predictor.py, so this does not replace model.joblib")
    rows.append({"id": EMB_ID, "setup": EMB_SETUP, "val_macro_f1": val_f1,
                 "val_accuracy": val_acc, "secs": secs, "pipe": None, "shippable": False})


## 6. Which run, if any, beat the baseline


In [ ]:
board = pd.DataFrame(rows).drop(columns=["pipe"]).sort_values("val_macro_f1", ascending=False)
board["delta_vs_baseline"] = (board.val_macro_f1 - BASELINE_F1).round(4)
print(f"baseline-01 val macro-F1 = {BASELINE_F1:.4f}")
board.round(4)


## 7. Ship a TF-IDF winner only

Leave `SHIP_IF_BETTER = False` until the table above shows a shippable row with a higher validation macro-F1 than baseline-01. Then set it to `True` and run this cell.

What it writes, matching notebook 01:

- refit the winning pipeline on train + validation
- sigmoid-calibrate the category head (the API confidence has to be a probability)
- refit the secondary-category and urgency heads with the **winner's** features
- replace `model/model.joblib` and `model/labels.json`

`exp-04-minilm` is never shipped from here. `app/predictor.py` only loads the three sklearn pipelines in the joblib dict.


In [ ]:
SHIP_IF_BETTER = False

shippable = [r for r in rows if r["shippable"]]
if not shippable:
    print("No TF-IDF run in memory. Run sections 3 and 4 first.")
else:
    winner = max(shippable, key=lambda r: r["val_macro_f1"])
    print(f"best TF-IDF this session: {winner['id']}  val macro-F1 {winner['val_macro_f1']:.4f}")
    if winner["val_macro_f1"] <= BASELINE_F1:
        print("It does not beat baseline-01. model/model.joblib stays as it is.")
    elif not SHIP_IF_BETTER:
        print("It beats the baseline. Set SHIP_IF_BETTER = True and re-run this cell to replace the file P4 loads.")
    else:
        kind = "svc" if "LinearSVC" in winner["setup"] else "logreg"
        C = float(re.search(r"C=([0-9.]+)", winner["setup"]).group(1))
        word = (1, 3) if "word(1,3)" in winner["setup"] else (1, 2)
        char = (3, 5) if "char_wb(3,5)" in winner["setup"] else (2, 5)
        full = pd.concat([train, val], ignore_index=True)
        Xf = full.model_text
        base_clf = make_clf(kind, C)
        cat_final = Pipeline([
            ("features", make_features(word, char)),
            ("clf", CalibratedClassifierCV(base_clf, method="sigmoid", cv=5)),
        ]).fit(Xf, full.category)
        sec_final = Pipeline([
            ("features", make_features(word, char)),
            ("clf", LogisticRegression(C=10, max_iter=2000, class_weight="balanced")),
        ]).fit(Xf, full.secondary_category.replace("", "none"))
        urg_final = Pipeline([
            ("features", make_features(word, char)),
            ("clf", LogisticRegression(C=10, max_iter=2000, class_weight="balanced")),
        ]).fit(Xf, full.is_urgent)

        CATEGORY_TO_TEAM = json.loads((MODEL_DIR / "labels.json").read_text(encoding="utf-8"))["category_to_team"]
        model_path = MODEL_DIR / "model.joblib"
        joblib.dump({"category": cat_final, "secondary": sec_final, "urgent": urg_final},
                    model_path, compress=3)
        sha = hashlib.sha256(model_path.read_bytes()).hexdigest()[:8]
        version = f"exp-tfidf-{sha}"
        labels = {
            "model_version": version,
            "sklearn_version": sklearn.__version__,
            "categories": [str(c) for c in cat_final.classes_],
            "category_to_team": CATEGORY_TO_TEAM,
            "refit_on_train_plus_val": True,
        }
        (MODEL_DIR / "labels.json").write_text(
            json.dumps(labels, indent=2, ensure_ascii=False), encoding="utf-8")
        print("shipped", version, "|", round(model_path.stat().st_size / 1e6, 1), "MB")
        print("Restart the API so app.predictor reloads this file.")
